In [1]:
!pip install tensorflow keras

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
base_dir = '/content/drive/MyDrive/skin_app'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

In [5]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input, # MobileNetV2's required preprocessing
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [6]:
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [7]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

train_class_names = list(train_generator.class_indices.keys())

validation_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    classes=train_class_names,
    shuffle=False
)

Found 2609 images belonging to 19 classes.
Found 895 images belonging to 19 classes.


In [8]:
train_classes = train_generator.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_classes),
    y=train_classes
)
class_weight_dict = dict(enumerate(class_weights))

In [9]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

In [10]:
base_model.trainable = False

In [11]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

In [12]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [13]:
lr_reduction = ReduceLROnPlateau(
    monitor='val_loss',
    patience=3,
    verbose=1,
    factor=0.5,
    min_lr=0.00001
)

In [14]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=8,
    verbose=1,
    restore_best_weights=True
)

In [ ]:
print("Starting real, honest AI training...")
history = model.fit(
    train_generator,
    epochs=30, # We can use more epochs because early stopping will catch it
    validation_data=validation_generator,
    class_weight=class_weight_dict,
    callbacks=[lr_reduction, early_stopping]
)

print("Training Complete! The best weights have been automatically restored.")

Starting real, honest AI training...
Epoch 1/30
82/82 ━━━━━━━━━━━━━━━━━━━━ 92s 1s/step - accuracy: 0.0997 - loss: 3.7758 - val_accuracy: 0.1430 - val_loss: 2.7431 - learning_rate: 0.0010
Epoch 2/30
82/82 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.1698 - loss: 3.1984 - val_accuracy: 0.1821 - val_loss: 2.7431 - learning_rate: 0.0010
Epoch 3/30
82/82 ━━━━━━━━━━━━━━━━━━━━ 87s 1s/step - accuracy: 0.2108 - loss: 2.7498 - val_accuracy: 0.1497 - val_loss: 2.8649 - learning_rate: 0.0010
Epoch 4/30
82/82 ━━━━━━━━━━━━━━━━━━━━ 88s 1s/step - accuracy: 0.2350 - loss: 2.5924 - val_accuracy: 0.1933 - val_loss: 2.7237 - learning_rate: 0.0010
Epoch 5/30
82/82 ━━━━━━━━━━━━━━━━━━━━ 88s 1s/step - accuracy: 0.2419 - loss: 2.4827 - val_accuracy: 0.2201 - val_loss: 2.6677 - learning_rate: 0.0010
Epoch 6/30
82/82 ━━━━━━━━━━━━━━━━━━━━ 88s 1s/step - accuracy: 0.2495 - loss: 2.3314 - val_accuracy: 0.2246 - val_loss: 2.6345 - learning_rate: 0.0010
Epoch 7/30
82/82 ━━━━━━━━━━━━━━━━━━━━ 88s 1s/step - accuracy: 0

In [ ]:
print("\n Phase 2: Unfreezing top layers of MobileNetV2...")

base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Starting Fine-Tuning...")
history_finetune = model.fit(
    train_generator,
    epochs=20,
    validation_data=validation_generator,
    class_weight=class_weight_dict,
    callbacks=[early_stopping]
)

model.save("/content/drive/MyDrive/skin_app/skin_model.keras")
print(" Model successfully saved for your web app!")